# Within-Strategy Trial-to-Trial Sequence Deviation

Measures how much each individual trial's firing sequence deviates from
its *own strategy's* reference sequence (Fast trials vs a Fast reference,
Slow trials vs a Slow reference), then compares Fast vs Slow per brain
region. Companion to the `### Monte Carlo ...` and `### Seq firing
deviation` sections of `2pAnalysis.ipynb` (which instead compare the two
*aggregate* sequences to each other). All logic lives in
`twop/seqdeviation.py`; this notebook only wires it up.

## Run this first to enable loading from relative paths

In [1]:
%load_ext autoreload
%autoreload 2
if "PKG" not in globals():
  import importlib, sys, pathlib # https://stackoverflow.com/a/50395128/11996983
  PKG = %pwd
  PKG = pathlib.Path(PKG)
  root_parent_level = 1
  root = PKG
  full_pkg = f"{root.name}"
  for _ in range(root_parent_level):
    root = root.parent
    full_pkg = f"{root.name}.{full_pkg}"
    MODULE_PATH = f"{root}{pathlib.os.path.sep}__init__.py"
    MODULE_NAME = f"{root.name}"
    spec = importlib.util.spec_from_file_location(MODULE_NAME, MODULE_PATH)
    module = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = module
    spec.loader.exec_module(module)
  __package__ = full_pkg

In [2]:
# Save plots with no embedded fonts (same as the other 2p notebooks)
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']

## Imports, flags, data load

In [ ]:
# Papermill parameters -- code/run_notebooks.py overrides these.
# Everything is off by default, so running the notebook writes nothing.
SAVE_FIGS = False           # write figures under results/
SAVE_DATA = False           # rewrite cached intermediate data under data/
PAPER_FIGURES_ONLY = False  # skip per-subject / per-session figures

In [3]:
import pickle
import pathlib
import pandas as pd

from .twop import seqdeviation as sd

MIN_PRCNT_ACTIVE = 5.0   # reference set: neurons active in > this % of a strategy's trials
MIN_ACTIVE_IN_TRIAL = 2  # a trial needs >= this many active reference neurons to score
RANK_METHOD = "dense"    # "dense" matches the MC / Seq-deviation sections of 2pAnalysis;
                         # peak sample spans ~30 frames so "dense" ties ~3 neurons/rank.
                         # Use "first" or "min" to give every neuron a distinct rank.

# Score shown in the part-C histograms:
#   "penalty"      -> global rank penalty (confounded by active-neuron count)
#   "norm_penalty" -> per-trial normalized disorder in 0..1, size-independent
HIST_SCORE_COL = "norm_penalty"
HIST_PER_TRIAL = True    # average within trial first (clean 0..1 for norm_penalty)

fig_save_prefix = "../results/2P/"
pathlib.Path(fig_save_prefix).mkdir(exist_ok=True)

In [4]:
# Fast = quantile 1 (impulsive), Slow = quantile 3 (deliberate). These are the
# same per-neuron max-firing dicts built/used in 2pAnalysis.ipynb "Sequence
# Extraction"; each row already carries the per-trial peak sample (`max_idxs`)
# keyed by `active_trial_numbers`, so nothing has to be recomputed from traces.
from .twop.dataload import dataPath, loadMaxFiringFrame

max_firing_q1_df = loadMaxFiringFrame("fast")
max_firing_q3_df = loadMaxFiringFrame("slow")
print("Fast neurons:", len(max_firing_q1_df),
      "| Slow neurons:", len(max_firing_q3_df))

Fast neurons: 1446 | Slow neurons: 1446


## Build the penalty dataframe (saved for reuse)

One row per trial x neuron, with four deviation scores:

- **`penalty`** (global) = `|ref_rank - observed_rank| / n_active_in_trial`.
  Gap-aware but confounded by active-neuron count (mean = displ / n^2).
- **`norm_penalty`** (local re-rank) = neurons re-ranked 1..n, footrule
  scaled to 0 (ref order) .. 1 (reversed), size-free but **gap-blind**.
- **`gap_norm_penalty`** = gap-aware footrule / this set's own max
  reversal -> 0..1, penalizes overtakes across larger reference gaps more.
- **`abs_gap_penalty`** = `|global displacement| / (n_ref_neurons-1)` ->
  gap-aware absolute displacement as a fraction of the whole sequence.

Also `local_ref_rank`, `local_observed_rank`, `n_active_in_trial`,
`n_ref_neurons`. Per-trial *mean* of any score is what the bars use.

In [5]:
# One row per (trial x neuron): within-strategy rank-deviation penalty.
# Fast trials are scored against the Fast reference sequence, Slow against Slow.
penalty_df = sd.build_penalty_df(
    max_firing_q1_df, max_firing_q3_df,
    min_prcnt_active=MIN_PRCNT_ACTIVE,
    min_active_in_trial=MIN_ACTIVE_IN_TRIAL,
    rank_method=RANK_METHOD,
)
if SAVE_DATA:
    penalty_df.to_pickle(dataPath("seq_within_deviation_df.pkl"))
print("penalty_df:", penalty_df.shape)
penalty_df.head()

penalty_df: (60679, 16)


,BrainRegion,ShortName,trace_id,trace_num,trial_strategy,TrialNumber,ref_rank,observed_rank,penalty,local_ref_rank,local_observed_rank,norm_penalty,gap_norm_penalty,abs_gap_penalty,n_active_in_trial,n_ref_neurons
0,LFC,GP4_23_s6_L50_D250_ALM,GP4_23_s6_L50_D250_ALM_50,50,Fast,192,16,16,0.000000,1,1,0.000000,0.000000,0.000000,2,71
1,LFC,GP4_23_s6_L50_D250_ALM,GP4_23_s6_L50_D250_ALM_0,0,Fast,192,27,27,0.000000,2,2,0.000000,0.000000,0.000000,2,71
2,LFC,GP4_23_s6_L50_D250_ALM,GP4_23_s6_L50_D250_ALM_42,42,Fast,203,4,20,0.941176,1,11,1.180556,1.271028,0.228571,17,71
3,LFC,GP4_23_s6_L50_D250_ALM,GP4_23_s6_L50_D250_ALM_6,6,Fast,203,13,13,0.000000,2,2,0.000000,0.000000,0.000000,17,71
4,LFC,GP4_23_s6_L50_D250_ALM,GP4_23_s6_L50_D250_ALM_71,71,Fast,203,15,17,0.117647,3,7,0.472222,0.158879,0.028571,17,71


## C. Per-session Fast vs Slow penalty histograms

In [6]:
# C. Per session: Fast vs Slow histograms of the selected score (one fig/session).
sd.plot_session_histograms(penalty_df, score_col=HIST_SCORE_COL,
                           per_trial=HIST_PER_TRIAL,
                           save_figs=SAVE_FIGS, fig_save_prefix=fig_save_prefix)

## D. Aggregate Fast vs Slow penalty, per brain region

In [7]:
# D. Aggregate Fast vs Slow per region. Compact table of all four scores first,
# then a bar figure for each:
#   penalty          global, gap-aware but confounded by active-neuron count
#   norm_penalty     local re-rank: size-free but gap-blind
#   gap_norm_penalty gap-aware, self-normalized (recommended)
#   abs_gap_penalty  gap-aware, absolute (fraction of the whole sequence)
display(sd.score_summary(penalty_df))
for _score in sd.ALL_SCORES:
    for _br in ["MFC", "LFC"]:
        sd.plot_region_bars(penalty_df, _br, score_col=_score,
                            save_figs=SAVE_FIGS, fig_save_prefix=fig_save_prefix)

penalty  norm_penalty  gap_norm_penalty  \
BrainRegion trial_strategy                                            
LFC         Fast              0.292         0.258             0.228   
            Slow              0.228         0.282             0.264   
MFC         Fast              0.309         0.271             0.240   
            Slow              0.211         0.275             0.250   

                            abs_gap_penalty  
BrainRegion trial_strategy                   
LFC         Fast                      0.055  
            Slow                      0.063  
MFC         Fast                      0.063  
            Slow                      0.059

## E. Do Fast and Slow use the *same* sequence? (cross analysis)

Build one strategy's firing order on the **commonly-active** neurons
(active in both strategies), then score both strategies' trials against
it: `matched` = the reference's own trials (the within-strategy floor),
`cross` = the other strategy's trials. If `cross` ~ `matched` the two
strategies share a sequence; if `cross` > `matched` they fire the common
neurons in a different order. Read `norm_penalty` (matched/cross trials
differ in active-neuron count, so the global `penalty` is confounded).

In [8]:
# For each reference strategy, build its firing order on the neurons active in
# BOTH strategies, then score `matched` (its own trials) and `cross` (the other
# strategy's trials) against that order. matched is the within-strategy floor;
# cross ~ matched -> same sequence, cross >> matched -> different sequence.
cross_fast_ref = sd.build_cross_penalty_df(max_firing_q1_df, max_firing_q3_df,
                                           reference="Fast", rank_method=RANK_METHOD)
cross_slow_ref = sd.build_cross_penalty_df(max_firing_q1_df, max_firing_q3_df,
                                           reference="Slow", rank_method=RANK_METHOD)
cross_df = pd.concat([cross_fast_ref, cross_slow_ref], ignore_index=True)
if SAVE_DATA:
    cross_df.to_pickle(dataPath("seq_cross_deviation_df.pkl"))
print("cross_df:", cross_df.shape,
      "| common neurons/session:",
      round(cross_df.groupby(["reference", "ShortName"]).n_ref_neurons.first()
            .groupby("reference").mean().mean(), 1))
cross_df.head()

cross_df: (119256, 18) | common neurons/session: 59.2


,BrainRegion,ShortName,trace_id,trace_num,trial_strategy,TrialNumber,ref_rank,observed_rank,penalty,local_ref_rank,local_observed_rank,norm_penalty,gap_norm_penalty,abs_gap_penalty,n_active_in_trial,n_ref_neurons,condition,reference
0,LFC,GP4_23_s6_L50_D250_ALM,GP4_23_s6_L50_D250_ALM_50,50,Fast,192,15,15,0.000000,1,1,0.000000,0.000000,0.000000,2,69,matched,Fast
1,LFC,GP4_23_s6_L50_D250_ALM,GP4_23_s6_L50_D250_ALM_0,0,Fast,192,26,26,0.000000,2,2,0.000000,0.000000,0.000000,2,69,matched,Fast
2,LFC,GP4_23_s6_L50_D250_ALM,GP4_23_s6_L50_D250_ALM_42,42,Fast,203,3,19,0.941176,1,11,1.180556,1.271028,0.235294,17,69,matched,Fast
3,LFC,GP4_23_s6_L50_D250_ALM,GP4_23_s6_L50_D250_ALM_6,6,Fast,203,12,12,0.000000,2,2,0.000000,0.000000,0.000000,17,69,matched,Fast
4,LFC,GP4_23_s6_L50_D250_ALM,GP4_23_s6_L50_D250_ALM_71,71,Fast,203,14,16,0.117647,3,7,0.472222,0.158879,0.029412,17,69,matched,Fast


In [9]:
# matched vs cross: compact table of all scores per reference, then bars for the
# size-free metrics. norm_penalty (gap-blind) vs gap_norm_penalty (gap-aware)
# shows whether the reference-gap magnitude changes the same-sequence answer.
for _cross, _ref in [(cross_fast_ref, "Fast"), (cross_slow_ref, "Slow")]:
    print(f"reference = {_ref}  (matched = {_ref} trials, cross = other)")
    display(sd.score_summary(_cross, group_col="condition"))
for _cross in [cross_fast_ref, cross_slow_ref]:
    for _score in [#"norm_penalty",
                   "gap_norm_penalty"]:
        for _br in ["MFC", "LFC"]:
            sd.plot_cross_bars(_cross, _br, score_col=_score,
                               save_figs=SAVE_FIGS, fig_save_prefix=fig_save_prefix)

reference = Fast  (matched = Fast trials, cross = other)


penalty  norm_penalty  gap_norm_penalty  \
BrainRegion condition                                            
LFC         cross        0.271         0.333             0.291   
            matched      0.292         0.257             0.227   
MFC         cross        0.301         0.333             0.306   
            matched      0.309         0.271             0.240   

                       abs_gap_penalty  
BrainRegion condition                   
LFC         cross                0.075  
            matched              0.056  
MFC         cross                0.087  
            matched              0.063

reference = Slow  (matched = Slow trials, cross = other)


penalty  norm_penalty  gap_norm_penalty  \
BrainRegion condition                                            
LFC         cross        0.338         0.303             0.284   
            matched      0.225         0.280             0.263   
MFC         cross        0.325         0.330             0.314   
            matched      0.211         0.273             0.250   

                       abs_gap_penalty  
BrainRegion condition                   
LFC         cross                0.066  
            matched              0.063  
MFC         cross                0.069  
            matched              0.060

### Significance: matched vs cross (paired, per session)

Test question: do the two distributions (reference-vs-itself = `matched`
and reference-vs-other-strategy = `cross`) differ? Sessions are the
independent unit and the two conditions are paired within a session, so
this is a **paired** test. Shapiro-Wilk on the per-session paired
differences decides **paired t-test** (normal) vs **Wilcoxon signed-rank**
(not); both two-sided. The table reports the normality result, the chosen
test, its statistic and p-value.

In [10]:
# Paired session-level significance test of matched vs cross. Sessions are the
# independent unit; matched and cross are paired within a session. Shapiro-Wilk
# on the per-session paired differences chooses the test (paired t if normal,
# else Wilcoxon signed-rank); two-sided -> "do the two distributions differ?".
CROSS_SIG_SCORES = ("norm_penalty", "gap_norm_penalty", "abs_gap_penalty")
cross_sig = pd.concat(
    [sd.cross_significance(cross_fast_ref, score_cols=CROSS_SIG_SCORES),
     sd.cross_significance(cross_slow_ref, score_cols=CROSS_SIG_SCORES)],
    ignore_index=True)
display(cross_sig.round(4))

,reference,BrainRegion,score,n_sessions,matched_mean,cross_mean,normality,shapiro_W,shapiro_p,normal,test,statistic,p_value,sig
0,Fast,LFC,norm_penalty,10,0.2574,0.3333,Shapiro-Wilk,0.8757,0.1163,True,paired t-test,7.9634,0.0000,***
1,Fast,MFC,norm_penalty,13,0.2715,0.3331,Shapiro-Wilk,0.9406,0.4652,True,paired t-test,3.1194,0.0089,**
2,Fast,LFC,gap_norm_penalty,10,0.2273,0.2907,Shapiro-Wilk,0.9196,0.3537,True,paired t-test,5.9628,0.0002,***
3,Fast,MFC,gap_norm_penalty,13,0.2401,0.3062,Shapiro-Wilk,0.9247,0.2905,True,paired t-test,3.3476,0.0058,**
4,Fast,LFC,abs_gap_penalty,10,0.0556,0.0752,Shapiro-Wilk,0.8644,0.0860,True,paired t-test,5.7304,0.0003,***
5,Fast,MFC,abs_gap_penalty,13,0.0632,0.0873,Shapiro-Wilk,0.8274,0.0147,False,Wilcoxon signed-rank,1.0000,0.0005,***
6,Slow,LFC,norm_penalty,10,0.2802,0.3030,Shapiro-Wilk,0.7921,0.0116,False,Wilcoxon signed-rank,11.0000,0.1055,ns
7,Slow,MFC,norm_penalty,13,0.2731,0.3295,Shapiro-Wilk,0.9597,0.7487,True,paired t-test,5.5944,0.0001,***
8,Slow,LFC,gap_norm_penalty,10,0.2626,0.2836,Shapiro-Wilk,0.6775,0.0005,False,Wilcoxon signed-rank,16.0000,0.2754,ns
9,Slow,MFC,gap_norm_penalty,13,0.2496,0.3144,Shapiro-Wilk,0.9276,0.3170,True,paired t-test,5.3344,0.0002,***


### Shuffle null: observed effect vs random within-trial ordering

A permutation test: reshuffle each trial's firing order many times, rebuild
the matched-vs-cross effect each time, and compare the observed effect to
that null. `chance_*` (the shuffle means) show both conditions sit far
below chance -- the sequences are real -- and `z_score` / `p_shuffle`
say whether the observed matched-vs-cross gap exceeds random ordering.
This treats **trials** as the unit (a chance floor); the paired test above
treats **sessions** as the unit (reproducibility across the replicate).

In [11]:
# Within-trial shuffle (permutation) null. For each of n_perm iterations, EVERY
# trial's observed firing order is randomly reshuffled, the per-trial score is
# recomputed from that trial's reference ranks, and the matched-vs-cross effect
# is rebuilt -> a null distribution of effects "under random ordering". The
# observed effect is compared to it (z-score + permutation p). chance_* are the
# shuffle means: both conditions sit far below chance (the sequences are real),
# and the observed effect is the small, reliable gap between them.
# NB this treats trials as the unit ("beyond random?"); the paired test above
# treats sessions as the unit ("reproducible across sessions?").
CROSS_SHUFFLE_SCORES = ("norm_penalty", "gap_norm_penalty", "abs_gap_penalty")
cross_shuffle = pd.concat(
    [sd.cross_shuffle_test(cross_fast_ref, score_cols=CROSS_SHUFFLE_SCORES, n_perm=2000),
     sd.cross_shuffle_test(cross_slow_ref, score_cols=CROSS_SHUFFLE_SCORES, n_perm=2000)],
    ignore_index=True)
display(cross_shuffle.round(4))

,reference,BrainRegion,score,n_perm,obs_matched,obs_cross,chance_matched,chance_cross,observed_effect,null_effect_mean,null_effect_std,z_score,p_shuffle_1sided,p_shuffle_2sided,sig
0,Fast,LFC,norm_penalty,2000,0.2574,0.3333,0.6582,0.6616,0.0759,0.0033,0.0083,8.7635,0.0005,0.0005,***
1,Fast,LFC,gap_norm_penalty,2000,0.2273,0.2907,0.6513,0.6506,0.0634,-0.0007,0.0091,7.0119,0.0005,0.0005,***
2,Fast,LFC,abs_gap_penalty,2000,0.0556,0.0752,0.1763,0.1778,0.0197,0.0015,0.0025,7.2657,0.0005,0.0005,***
3,Fast,MFC,norm_penalty,2000,0.2715,0.3331,0.6618,0.6642,0.0616,0.0024,0.0063,9.3211,0.0005,0.0005,***
4,Fast,MFC,gap_norm_penalty,2000,0.2401,0.3062,0.6568,0.6573,0.0661,0.0005,0.0068,9.6665,0.0005,0.0005,***
5,Fast,MFC,abs_gap_penalty,2000,0.0632,0.0873,0.1805,0.1915,0.0241,0.0109,0.0020,6.4836,0.0005,0.0005,***
6,Slow,LFC,norm_penalty,2000,0.2802,0.3030,0.6616,0.6582,0.0228,-0.0034,0.0083,3.1448,0.0020,0.0040,**
7,Slow,LFC,gap_norm_penalty,2000,0.2626,0.2836,0.6634,0.6592,0.0210,-0.0042,0.0090,2.8131,0.0035,0.0065,**
8,Slow,LFC,abs_gap_penalty,2000,0.0630,0.0659,0.1711,0.1636,0.0029,-0.0075,0.0023,4.5967,0.0005,0.0005,***
9,Slow,MFC,norm_penalty,2000,0.2731,0.3295,0.6642,0.6618,0.0565,-0.0025,0.0064,9.1905,0.0005,0.0005,***


## F. Observed disorder as a controlled-shuffle level

Parts C-E compare each strategy's disorder against a single fully-random
shuffle (chance ~0.667). Here the shuffle is made *continuous*: a Mallows
perturbation walks the firing order from the reference sequence (0%, no rank
deviation) to its exact reverse (100%), and the gap-aware disorder is measured
at each level. Reading each session's observed Fast/Slow disorder back off that
calibration expresses it as an equivalent shuffle level (e.g. "as disordered as
inverting ~20% of the neuron pairs"). **50% on this axis is exactly the
uniform-random case**, so x > 50% would mean "more reversed than chance".

Two figures per region (Fast reference, Slow reference).

In [12]:
# F. Controlled-shuffle calibration (Mallows: reference order -> reversed).
# Expensive (Mallows sampling per active-set size x 32 levels x reps), so build
# once for both references and cache to a pickle like penalty_df. Lowering
# SHUFFLE_CAL_REPS only thins the grey background cloud, not the curve accuracy.
SHUFFLE_CAL_REPS = 100
SHUFFLE_CAL_SEED = 0
SHUFFLE_CAL_SCORE = "gap_norm_penalty"   # y-axis: mean per-trial gap-aware disorder
# Stop the shuffle scale at 50% = chance (no reversed region / artificial sign-flip):
# every session sits well below chance (max ~35%), so this only drops the unused
# 50-100% half and leaves all shuffle-%% values unchanged.
SHUFFLE_LEVELS_50 = sd.SHUFFLE_LEVELS[sd.SHUFFLE_LEVELS <= 0.5]

calib_fast_ref = sd.shuffle_calibration(cross_fast_ref, score_col=SHUFFLE_CAL_SCORE,
                                        levels=SHUFFLE_LEVELS_50,
                                        n_rep=SHUFFLE_CAL_REPS, seed=SHUFFLE_CAL_SEED)
calib_slow_ref = sd.shuffle_calibration(cross_slow_ref, score_col=SHUFFLE_CAL_SCORE,
                                        levels=SHUFFLE_LEVELS_50,
                                        n_rep=SHUFFLE_CAL_REPS, seed=SHUFFLE_CAL_SEED)
calib_df = pd.concat([calib_fast_ref, calib_slow_ref], ignore_index=True)
if SAVE_DATA:
    calib_df.to_pickle(dataPath("seq_shuffle_calibration_df.pkl"))
print("calib_df:", calib_df.shape)
calib_df.head()


shuffle calibration (Fast ref, gap_norm_penalty):   0%|          | 0/23 [00:00<?, ?it/s]

shuffle calibration (Slow ref, gap_norm_penalty):   0%|          | 0/23 [00:00<?, ?it/s]

calib_df: (101200, 6)


,reference,BrainRegion,ShortName,level,rep,score
0,Fast,LFC,GP4_23_s6_L50_D250_ALM,0.0,0,0.0
1,Fast,LFC,GP4_23_s6_L50_D250_ALM,0.0,1,0.0
2,Fast,LFC,GP4_23_s6_L50_D250_ALM,0.0,2,0.0
3,Fast,LFC,GP4_23_s6_L50_D250_ALM,0.0,3,0.0
4,Fast,LFC,GP4_23_s6_L50_D250_ALM,0.0,4,0.0


In [13]:
# One figure per (reference, region): each session's observed Fast (red) and
# Slow (gold) disorder placed on the shuffle axis by inverting that session's
# own calibration curve; big dots = mean +/- sem over sessions.
for _cross, _cal in [(cross_fast_ref, calib_fast_ref),
                     (cross_slow_ref, calib_slow_ref)]:
    for _br in ["MFC", "LFC"]:
        sd.plot_shuffle_calibration(_cross, _br, _cal, score_col=SHUFFLE_CAL_SCORE,
                                    save_figs=SAVE_FIGS, fig_save_prefix=fig_save_prefix)

### F2. "Same-sequence?" panels on the shuffle-level axis (random = 50%)

The four cross panels (MFC/LFC x Fast-ref/Slow-ref), with the gap-aware disorder
remapped to its equivalent **Mallows shuffle level** -- the % of neuron pairs
inverted vs the reference. `0% = reference order`, **`50% = random`**,
`100% = fully reversed`; a value well below 50% means the sequence is still
ordered like the reference (not reversed).

Each panel's **left bar = within** (the reference strategy's own trials vs its
own reference) and **right bar = cross** (the other strategy's trials vs that
reference), both on the neurons active in **both** strategies. Each panel is
inverted through its own reference's calibration (`calib_fast_ref` /
`calib_slow_ref`), so the bars match the F3 calibration panels exactly. The remap
is *monotone*, so it preserves every ordering and significance result.

In [14]:
# The four "same-sequence?" panels on the shuffle-level axis (0 = reference order,
# 50 = random, 100 = reversed). Each panel is inverted through its OWN reference's
# calibration (calib_fast_ref / calib_slow_ref), so the bars match the F3 panels
# exactly (MFC Fast-ref within = 16.1%). Left bar = within (reference strategy's own
# trials), right bar = cross (the other strategy vs that reference).
for _cross, _cal in [(cross_fast_ref, calib_fast_ref),
                     (cross_slow_ref, calib_slow_ref)]:
    for _br in ["MFC", "LFC"]:
        sd.plot_shuffle_level_bars(_cross, _br, _cal, group_col="condition",
                                   save_figs=SAVE_FIGS, fig_save_prefix=fig_save_prefix)

### F3. The F calibration, recentered so random = 50% (matches F2)

The same F1 figure, but the **y-axis is passed through the F2 Mallows
recentering** (raw disorder -> shuffle-equivalent %), so random sits at 50% on
*both* axes. Because that recentering uses this very calibration, the curve
collapses to the straight diagonal `y = x` and random lands dead-centre at
`(50, 50)`; the Fast/Slow dots sit on the diagonal at their F2 shuffle %. It is
the linearized consistency view of F1 on the shared 0-100 scale.

In [15]:
# F3: F1 recentered (Mallows, same as F2) so random = 50% on the y-axis too.
# The curve becomes y = x by construction; dots land on it at their shuffle %.
for _cross, _cal in [(cross_fast_ref, calib_fast_ref),
                     (cross_slow_ref, calib_slow_ref)]:
    for _br in ["MFC", "LFC"]:
        sd.plot_shuffle_calibration(_cross, _br, _cal, score_col=SHUFFLE_CAL_SCORE,
                                    recenter=True,
                                    save_figs=SAVE_FIGS, fig_save_prefix=fig_save_prefix)

### F4. Replay a single shuffle, step by step (interactive)

Pick a trial and a shuffle level, press **Run / record** to capture the exact
repeated-insertion (RIM) construction of one permutation, then drag the **step**
slider to watch it built one neuron at a time. The top plot shows how the level
sets `phi`; the track shows the reference order vs the growing shuffled order,
with each inserted neuron highlighted and its jump (inversions) annotated.

*Interactive (ipywidgets) — runs in the conda `py312` kernel; it will not render
under the headless runner. Tip: the trial dropdown shows each trial's neuron count
`n` — pick a small `n` for the clearest step-through.*

In [ ]:
# Interactive step-by-step replay of one Mallows shuffle (see shuffle_replay.py).
from .twop import shuffle_replay
shuffle_replay.show_shuffle_replay(cross_df)